# Лабораторная работа №5. Градиентный бустинг

Чибугаев Илья Алексеевич, вариант 22.

## Данные и метрики

Для классификации взят [Ionosphere](https://archive.ics.uci.edu/dataset/52/ionosphere): 351 радиосигнал и 34 признака. Класс `bad=1` означает сигнал без структуры в ионосфере. Это задача автоматической проверки радиосигналов.

Для регрессии взят [Concrete Compressive Strength](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength): 1030 составов бетона и 8 признаков. По составу и возрасту нужно оценить прочность в МПа. Такой прогноз полезен перед лабораторными испытаниями смеси.

Бустинг добавляет деревья последовательно. Каждое следующее исправляет ошибку текущего ансамбля: остаток в регрессии или градиент логистической потери в классификации.

Train и test соотносятся как 80/20, `random_state=22`. Классы при разбиении сохраняются в одинаковой пропорции. Подбор параметров идет только на train.

В классификации основной показатель: ROC-AUC. Он не зависит от выбранного порога. F1 показывает качество для `bad`, accuracy дает общую долю верных ответов. В регрессии основной показатель: RMSE в МПа. MAE проще интерпретировать как типичную ошибку, а R² показывает долю объясненной дисперсии.


## Подготовка

Общий код вынесен в `src`, чтобы не повторять его в каждой работе.

In [1]:
# Подключаю общие функции и настраиваю вывод таблиц.
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.experiments import TaskExperiment

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
plt.style.use("seaborn-v0_8-whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"Корень проекта: {ROOT}")

Корень проекта: /Users/finleyl/Documents/kimi/tasks/2026-11-10/20-02-04-6933ea61/ml-labs-1-5


## Проверка данных

Смотрю размеры выборок, пропуски и целевые переменные. Эти разбиения используются во всех работах.

In [2]:
# Загружаю данные и проверяю размеры выборок.
experiments = {task: TaskExperiment(5, task) for task in ("classification", "regression")}
for task, experiment in experiments.items():
    data = experiment.data
    print(f"{task}: {data.name}")
    print(data.practical_task)
    print(f"train={data.X_train.shape}, test={data.X_test.shape}, пропуски={int(data.X.isna().sum().sum())}")
    if task == "classification":
        print(f"Доля bad: train={data.y_train.mean():.3f}, test={data.y_test.mean():.3f}")
    else:
        print(data.y.describe().to_string())
    print()

classification: Ionosphere (UCI)
По сигналам фазированной антенной решетки отличать радиосигналы со структурой в ионосфере от проходящих сигналов.
train=(280, 34), test=(71, 34), пропуски=0
Доля bad: train=0.361, test=0.352

regression: Concrete Compressive Strength (UCI)
По составу смеси и возрасту бетона прогнозировать прочность на сжатие в МПа для предварительной оценки рецептуры.
train=(824, 8), test=(206, 8), пропуски=0
count    1030.0000
mean       35.8178
std        16.7057
min         2.3318
25%        23.7071
50%        34.4428
75%        46.1363
max        82.5992



## Бейзлайн sklearn

Беру модель с параметрами по умолчанию. Медианный imputer оставлен как страховка, хотя пропусков в данных нет.

In [3]:
# Обучаю базовые модели sklearn.
for experiment in experiments.values():
    experiment.fit_sklearn_baseline()
baseline_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(baseline_results.drop(columns=["params"]))

classification | sklearn | baseline: accuracy=0.9437, f1=0.9167, roc_auc=0.9957; 0.09 s
regression | sklearn | baseline: rmse=5.2916, mae=3.9903, r2=0.9062; 0.06 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,5,Градиентный бустинг,classification,sklearn,baseline,0.9437,0.9167,0.9957,NaN,NaN,NaN,0.0888
1,5,Градиентный бустинг,regression,sklearn,baseline,NaN,NaN,NaN,5.2916,3.9903,0.9062,0.0563


## Подбор параметров

Подбираю скорость обучения, число деревьев и глубину. Маленький шаг обычно требует большего числа деревьев, поэтому эти параметры рассматриваются вместе.

Использую пятикратную CV. Для классификации выбираю по ROC-AUC, для регрессии по RMSE.

In [4]:
# Подбираю параметры только на обучающей выборке.
for experiment in experiments.values():
    experiment.fit_sklearn_improved()
    print(f"{{experiment.task}}: {{experiment.search.best_params_}}")
    display(experiment.cv_table().head())

classification | sklearn | improved: accuracy=0.9437, f1=0.9130, roc_auc=0.9974; 1.83 s
{experiment.task}: {experiment.search.best_params_}


,params,cv_roc_auc,std_roc_auc,mean_fit_time
0,"{'model__learning_rate': 0.1, 'model__max_dept...",0.9630,0.0189,0.1326
1,"{'model__learning_rate': 0.1, 'model__max_dept...",0.9601,0.0220,0.0679
2,"{'model__learning_rate': 0.1, 'model__max_dept...",0.9597,0.0226,0.1537
3,"{'model__learning_rate': 0.1, 'model__max_dept...",0.9551,0.0273,0.1004
4,"{'model__learning_rate': 0.03, 'model__max_dep...",0.9549,0.0260,0.1445


regression | sklearn | improved: rmse=4.6679, mae=3.4299, r2=0.9270; 0.47 s
{experiment.task}: {experiment.search.best_params_}


,params,cv_rmse,std_rmse,mean_fit_time
0,"{'model__learning_rate': 0.1, 'model__max_dept...",5.0211,0.4468,0.1055
1,"{'model__learning_rate': 0.1, 'model__max_dept...",5.3626,0.4882,0.0942
2,"{'model__learning_rate': 0.1, 'model__max_dept...",5.4062,0.4424,0.0633
3,"{'model__learning_rate': 0.03, 'model__max_dep...",5.8388,0.4061,0.1412
4,"{'model__learning_rate': 0.1, 'model__max_dept...",5.9364,0.4480,0.0532


## Что дала настройка

Сравниваю метрики до и после. Для ROC-AUC, F1, accuracy и R² рост означает улучшение, для RMSE и MAE наоборот.

In [5]:
# Считаю разницу метрик между базовой и настроенной моделями.
metric_names = {"classification": ["roc_auc", "f1", "accuracy"], "regression": ["rmse", "mae", "r2"]}
for task, experiment in experiments.items():
    rows = experiment.results.set_index("stage")
    metrics = metric_names[task]
    comparison = rows.loc[["baseline", "improved"], metrics].T
    comparison["change"] = comparison["improved"] - comparison["baseline"]
    print(task)
    display(comparison)

classification


stage,baseline,improved,change
roc_auc,0.9957,0.9974,0.0017
f1,0.9167,0.9130,-0.0036
accuracy,0.9437,0.9437,0.0000


regression


stage,baseline,improved,change
rmse,5.2916,4.6679,-0.6238
mae,3.9903,3.4299,-0.5604
r2,0.9062,0.9270,0.0208


## Своя реализация

В своей версии базовые модели являются регрессионными деревьями. Их прогнозы складываются с коэффициентом `learning_rate`.

Код лежит в `src/models/`. Сначала обучаю базовую версию на тех же данных, что и sklearn.

In [6]:
# Обучаю базовые версии собственных моделей.
for experiment in experiments.values():
    experiment.fit_custom_baseline()
custom_baseline_results = pd.concat([
    experiment.results[experiment.results["implementation"] == "from_scratch"]
    for experiment in experiments.values()
], ignore_index=True)
display(custom_baseline_results.drop(columns=["params"]))

classification | from_scratch | baseline: accuracy=0.9296, f1=0.8936, roc_auc=0.9904; 0.36 s
regression | from_scratch | baseline: rmse=5.3136, mae=3.9920, r2=0.9055; 0.15 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,5,Градиентный бустинг,classification,from_scratch,baseline,0.9296,0.8936,0.9904,NaN,NaN,NaN,0.3558
1,5,Градиентный бустинг,regression,from_scratch,baseline,NaN,NaN,NaN,5.3136,3.992,0.9055,0.1537


## Своя модель после настройки

Переношу препроцессинг и параметры, выбранные на CV. Ограничение по числу деревьев для своего леса указано выше.

In [7]:
# Применяю найденные параметры к своим моделям.
for experiment in experiments.values():
    experiment.fit_custom_improved()
all_lab_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(all_lab_results.drop(columns=["params"]))

classification | from_scratch | improved: accuracy=0.9718, f1=0.9583, roc_auc=0.9952; 0.36 s


regression | from_scratch | improved: rmse=4.6195, mae=3.3796, r2=0.9285; 0.31 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,5,Градиентный бустинг,classification,sklearn,baseline,0.9437,0.9167,0.9957,NaN,NaN,NaN,0.0888
1,5,Градиентный бустинг,classification,sklearn,improved,0.9437,0.9130,0.9974,NaN,NaN,NaN,1.8314
2,5,Градиентный бустинг,classification,from_scratch,baseline,0.9296,0.8936,0.9904,NaN,NaN,NaN,0.3558
3,5,Градиентный бустинг,classification,from_scratch,improved,0.9718,0.9583,0.9952,NaN,NaN,NaN,0.3582
4,5,Градиентный бустинг,regression,sklearn,baseline,NaN,NaN,NaN,5.2916,3.9903,0.9062,0.0563
5,5,Градиентный бустинг,regression,sklearn,improved,NaN,NaN,NaN,4.6679,3.4299,0.9270,0.4721
6,5,Градиентный бустинг,regression,from_scratch,baseline,NaN,NaN,NaN,5.3136,3.9920,0.9055,0.1537
7,5,Градиентный бустинг,regression,from_scratch,improved,NaN,NaN,NaN,4.6195,3.3796,0.9285,0.3100


## Результаты

Сохраняю все четыре варианта для каждой задачи. Вывод зависит от полученных чисел: если настройка не помогла, это будет видно.

In [8]:
# Сохраняю таблицу и вывожу изменение основной метрики.
output_path = ROOT / "results" / "lab05_results.csv"
output_path.parent.mkdir(exist_ok=True)
all_lab_results.to_csv(output_path, index=False)

def verdict(change, higher_is_better):
    if np.isclose(change, 0):
        return "не изменился"
    improved = change > 0 if higher_is_better else change < 0
    return "улучшился" if improved else "ухудшился"

for task, experiment in experiments.items():
    frame = experiment.results.set_index(["implementation", "stage"])
    metrics = metric_names[task]
    primary = "roc_auc" if task == "classification" else "rmse"
    higher_is_better = primary != "rmse"
    sklearn_base = frame.loc[("sklearn", "baseline"), primary]
    sklearn_best = frame.loc[("sklearn", "improved"), primary]
    custom_base = frame.loc[("from_scratch", "baseline"), primary]
    custom_best = frame.loc[("from_scratch", "improved"), primary]
    print(f"{task}, sklearn: {primary} {sklearn_base:.4f} → {sklearn_best:.4f}, "
          f"{verdict(sklearn_best - sklearn_base, higher_is_better)}.")
    print(f"{task}, своя версия: {primary} {custom_base:.4f} → {custom_best:.4f}, "
          f"{verdict(custom_best - custom_base, higher_is_better)}.")
    print(f"Параметры sklearn: {experiment.search.best_params_}")
print(f"Файл: {output_path.relative_to(ROOT)}")

classification, sklearn: roc_auc 0.9957 → 0.9974, улучшился.
classification, своя версия: roc_auc 0.9904 → 0.9952, улучшился.
Параметры sklearn: {'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 200}
regression, sklearn: rmse 5.2916 → 4.6679, улучшился.
regression, своя версия: rmse 5.3136 → 4.6195, улучшился.
Параметры sklearn: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}
Файл: results/lab05_results.csv
